# 04 Train TempSim

Notebook này smoke-test phần retrieval theo paper cho Người 2 trên artifact `handover_data`. Pha 1 vẫn chỉ dùng retrieval payload trực tiếp; `similar_case_report` sẽ nối sau.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)


C:\Users\DELL\Downloads\codedll\HealthDataMining


In [4]:
%pip install torch pyarrow

from pathlib import Path

import pyarrow.parquet as pq
import torch

from src.data.dataset import collate_batch
from src.models.patient_state_encoder import PatientStateEncoder
from src.retrieval.dynamic_graph import build_edge_artifact
from src.retrieval.memory_bank import MemoryBank, build_last_visit_queries
from src.retrieval.topk_retriever import retrieve_patient_neighbors, retrieve_personal_history
from src.utils.io import read_json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
HANDOVER_ROOT = WORKSPACE_ROOT / 'handover_data'
if not HANDOVER_ROOT.exists():
    raise FileNotFoundError(
        f'Missing extracted handover_data at {HANDOVER_ROOT}. Extract handover_data.tar.gz first.'
    )
print('PROJECT_ROOT =', PROJECT_ROOT)
print('HANDOVER_ROOT =', HANDOVER_ROOT)


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


PROJECT_ROOT = C:\Users\DELL\Downloads\codedll\HealthDataMining
HANDOVER_ROOT = C:\Users\DELL\Downloads\codedll\handover_data


In [5]:
def load_records(split: str, limit: int = 8):
    split_dir = HANDOVER_ROOT / 'processed' / split
    shard_paths = sorted(split_dir.glob('*.parquet'))
    if not shard_paths:
        raise FileNotFoundError(f'No parquet shards found under {split_dir}')
    records = []
    for shard_path in shard_paths:
        table = pq.read_table(shard_path)
        for record in table.to_pylist():
            record['drug_vocab_size'] = int(record.get('drug_vocab_size', 0))
            records.append(record)
            if len(records) >= limit:
                return records
    return records


train_records = load_records('train', limit=8)
len(train_records), train_records[0]['stay_id'], train_records[0]['split']

(8, 30001336, 'train')

In [6]:
batch = collate_batch(train_records)

vocab_root = HANDOVER_ROOT / 'vocab'
diagnosis_vocab = read_json(vocab_root / 'diagnosis_vocab.json')
procedure_vocab = read_json(vocab_root / 'procedure_vocab.json')
drug_vocab = read_json(vocab_root / 'drug_vocab.json')

model = PatientStateEncoder(
    diagnosis_vocab_size=diagnosis_vocab['size'],
    procedure_vocab_size=procedure_vocab['size'],
    drug_vocab_size=drug_vocab['size'],
    num_lab_features=batch['lab_values'].shape[-1],
    num_vital_features=batch['vital_values'].shape[-1],
)

with torch.no_grad():
    encoder_out = model(batch)

encoder_out['state_sequence'].shape

torch.Size([8, 17, 128])

In [7]:
memory_bank = MemoryBank.build_from_batch(train_records, encoder_out, split='train')
query_visit_states, query_metadata = build_last_visit_queries(train_records[:2], {
    'state_sequence': encoder_out['state_sequence'][:2],
    'visit_mask': encoder_out['visit_mask'][:2],
}, split='train')

personal_history = retrieve_personal_history(
    query_visit_states,
    query_metadata,
    memory_bank,
    top_k=3,
    temporal_decay_alpha=0.05,
)

retrieval_payload = retrieve_patient_neighbors(
    query_visit_states,
    query_metadata,
    memory_bank,
    top_k=3,
    temporal_decay_alpha=0.05,
    backend='bruteforce',
)

edge_artifact = build_edge_artifact(retrieval_payload)
personal_history, retrieval_payload, edge_artifact['edges'][:3]

({'indices': tensor([[ 1,  0],
          [ 3, -1]]),
  'scores': tensor([[1.5599, 1.0952],
          [1.2480,   -inf]])},
 {'query_stay_ids': [30001336, 30001446],
  'query_split': 'train',
  'query_visit_indices': tensor([2, 1]),
  'neighbor_indices': tensor([[ 4, 11,  5],
          [ 2, 11,  5]]),
  'neighbor_scores': tensor([[2.6849e-01, 2.7957e-18, 0.0000e+00],
          [2.6849e-01, 6.7923e-18, 0.0000e+00]]),
  'neighbor_static_scores': tensor([[0.6713, 0.7390, 0.5142],
          [0.6713, 0.6469, 0.5004]]),
  'neighbor_time_gaps_days': tensor([[2.2188e+01, 8.0800e+02, 2.7245e+04],
          [2.2188e+01, 7.8581e+02, 2.7268e+04]]),
  'neighbor_subject_ids': tensor([[16513856, 16235911, 14311522],
          [12168737, 16235911, 14311522]]),
  'neighbor_hadm_ids': tensor([[24463832, 28956560, 24622512],
          [29283664, 28956560, 24622512]]),
  'neighbor_stay_ids': tensor([[30001446, 30003306, 30002548],
          [30001336, 30003306, 30002548]]),
  'matched_visit_indices': tensor